임포트

In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

경로 설정

In [2]:
cwd = Path(os.getcwd()).resolve()

def find_project_root(start: Path, max_up: int = 6) -> Path:
    cur = start
    for _ in range(max_up):
        if (cur / "data" / "processed").exists():
            return cur
        cur = cur.parent
    return start.parent

PROJECT_ROOT = find_project_root(cwd)
PROCESSED = PROJECT_ROOT / "data" / "processed"

PITCH_PATH   = PROCESSED / "pitch_umap_cluster.parquet"
PITCHER_PROF = PROCESSED / "pitcher_profiles.parquet"
BATTER_PROF  = PROCESSED / "batter_profiles.parquet"

OUT_PITCH_LEVEL = PROCESSED / "matchup_pitch_level.parquet"
OUT_PAIR_LEVEL  = PROCESSED / "matchup_pair_level.parquet"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PITCH_PATH:", PITCH_PATH, "exists?", PITCH_PATH.exists())
print("PITCHER_PROF:", PITCHER_PROF, "exists?", PITCHER_PROF.exists())
print("BATTER_PROF:", BATTER_PROF, "exists?", BATTER_PROF.exists())
print("OUT_PITCH_LEVEL:", OUT_PITCH_LEVEL)
print("OUT_PAIR_LEVEL:", OUT_PAIR_LEVEL)

PROJECT_ROOT: C:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess
PITCH_PATH: C:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\data\processed\pitch_umap_cluster.parquet exists? True
PITCHER_PROF: C:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\data\processed\pitcher_profiles.parquet exists? True
BATTER_PROF: C:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\data\processed\batter_profiles.parquet exists? True
OUT_PITCH_LEVEL: C:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\data\processed\matchup_pitch_level.parquet
OUT_PAIR_LEVEL: C:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\data\processed\matchup_pair_level.parquet


데이터 로드

In [3]:
dfp = pd.read_parquet(PITCH_PATH)
pp  = pd.read_parquet(PITCHER_PROF)
bp  = pd.read_parquet(BATTER_PROF)

print("pitch-level:", dfp.shape)
print("pitcher_profiles:", pp.shape)
print("batter_profiles:", bp.shape)

dfp.head(3)

pitch-level: (719509, 40)
pitcher_profiles: (799, 2379)
batter_profiles: (674, 15)


,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,batter,pitcher,events,description,zone,...,bat_score_diff,arm_angle,description_group,events_group,base_state,count,umap_x,umap_y,pitch_cluster_local,pitch_cluster_id
0,FF,2025-03-28,97.8,-2.47,5.80,656811,592332,field_out,hit_into_play,11.0,...,-3,38.4,inplay,out,0,1-2,15.180676,1.919086,0,592332_0
1,SL,2025-03-28,85.8,-2.45,5.63,656811,592332,NaN,foul,5.0,...,-3,35.5,foul,other,0,1-2,-2.767725,13.609803,1,592332_1
2,FF,2025-03-28,95.8,-2.33,5.89,656811,592332,NaN,foul,1.0,...,-3,40.6,foul,other,0,1-1,14.007529,1.559027,0,592332_0


최소 컬럼 체크 + 라벨 준비

In [4]:
# 필수 컬럼
need = ["pitcher", "batter", "pitch_type", "balls", "strikes"]
missing = [c for c in need if c not in dfp.columns]
if missing:
    raise KeyError(f"Missing columns in pitch-level data: {missing}")

# 가능한 라벨 후보
label_candidates = [
    "events_group",       # 01에서 만들었다면 최고
    "events",             # 원본 events
    "description_group",  # 01에서 만들었다면 좋음
    "description",        # 원본 description
]

LABEL_COL = None
for c in label_candidates:
    if c in dfp.columns:
        LABEL_COL = c
        break

if LABEL_COL is None:
    raise KeyError("No label column found among events_group/events/description_group/description")

print("Using LABEL_COL =", LABEL_COL)

# 기본 파생 label (간단한 분류로도 하나 만들어두면 편함)
# - if LABEL_COL이 events면 hit/out/walk 비슷하게
if LABEL_COL in ["events_group", "events"]:
    # events_group이 있으면 그대로 쓰고, events면 매핑
    if LABEL_COL == "events":
        EVENT_TO_GROUP = {
            "single": "hit", "double": "hit", "triple": "hit", "home_run": "hit",
            "walk": "walk", "intent_walk": "walk", "hit_by_pitch": "walk",
            "strikeout": "out", "strikeout_double_play": "out",
            "field_out": "out", "force_out": "out", "double_play": "out",
            "grounded_into_double_play": "out",
            "field_error": "reached",
        }
        dfp["label_events_group"] = dfp["events"].map(EVENT_TO_GROUP).fillna("other")
    else:
        dfp["label_events_group"] = dfp["events_group"].fillna("other")
else:
    dfp["label_events_group"] = "other"

# description을 ball/strike/inplay로 단순화한 라벨도 만들어둠
if "description_group" in dfp.columns:
    dfp["label_desc_group"] = dfp["description_group"].fillna("other")
else:
    # 원본 description 기반 최소 매핑
    DESC_TO_GROUP = {
        "called_strike": "strike",
        "swinging_strike": "strike",
        "swinging_strike_blocked": "strike",
        "missed_bunt": "strike",
        "foul": "foul",
        "foul_tip": "foul",
        "foul_bunt": "foul",
        "ball": "ball",
        "blocked_ball": "ball",
        "pitchout": "ball",
        "intent_ball": "ball",
        "hit_into_play": "inplay",
        "hit_into_play_score": "inplay",
        "hit_into_play_no_out": "inplay",
    }
    if "description" in dfp.columns:
        dfp["label_desc_group"] = dfp["description"].map(DESC_TO_GROUP).fillna("other")
    else:
        dfp["label_desc_group"] = "other"

dfp[["label_events_group","label_desc_group"]].value_counts().head(10)

Using LABEL_COL = events_group


label_events_group  label_desc_group
other               ball                241696
                    strike              158450
                    foul                134875
out                 inplay               82108
hit                 inplay               40493
out                 strike               38365
walk                ball                 14969
out                 foul                  2924
other               inplay                2619
walk                other                 1953
Name: count, dtype: int64

A) Pitch-level 학습 테이블 만들기

pitcher/batter 프로필 조인용 컬럼 정리

In [5]:
# profiles에서 조인키만 남기고, feature 컬럼만 사용
if "pitcher" not in pp.columns:
    raise KeyError("pitcher_profiles must have 'pitcher' column")
if "batter" not in bp.columns:
    raise KeyError("batter_profiles must have 'batter' column")

# 중복/불필요한 텍스트 컬럼 제거(있으면)
pp_use = pp.copy()
bp_use = bp.copy()

# pitcher_profiles에서 model에 넣기 좋은 feature들만 남김
drop_pp = [c for c in ["note"] if c in pp_use.columns]
if drop_pp:
    pp_use = pp_use.drop(columns=drop_pp)

# batter_profiles도 텍스트가 있으면 제외(stand_mode는 모델에 써도 되지만 일단 유지)
# 필요하면 one-hot 처리 단계에서 처리
print("pp_use cols:", len(pp_use.columns))
print("bp_use cols:", len(bp_use.columns))

pp_use cols: 2379
bp_use cols: 15


pitch-level 테이블 생성 (조인 + 최소 feature 선택)

In [6]:
# 모델 입력에 기본으로 넣을 pitch-level feature (필요 시 추가/삭제)
base_pitch_features = [
    "pitcher","batter","game_date",
    "pitch_type",
    "balls","strikes",
    "plate_x","plate_z",
    "release_speed","release_spin_rate",
    "pfx_x","pfx_z",
    "release_pos_x","release_pos_z",
    "release_extension","arm_angle",
    "outs_when_up","inning","inning_topbot","bat_score_diff",
    "on_1b","on_2b","on_3b",
    "pitch_cluster_id","pitch_cluster_local",
    "umap_x","umap_y",
    "launch_speed","launch_angle",
]

base_pitch_features = [c for c in base_pitch_features if c in dfp.columns]
print("base_pitch_features used:", base_pitch_features)

# 조인
pitch_level = dfp[base_pitch_features + ["label_events_group","label_desc_group"]].copy()

pitch_level = pitch_level.merge(pp_use, on="pitcher", how="left", suffixes=("","_pp"))
pitch_level = pitch_level.merge(bp_use, on="batter", how="left", suffixes=("","_bp"))

print("pitch_level:", pitch_level.shape)
pitch_level.head(3)

base_pitch_features used: ['pitcher', 'batter', 'game_date', 'pitch_type', 'balls', 'strikes', 'plate_x', 'plate_z', 'release_speed', 'release_spin_rate', 'pfx_x', 'pfx_z', 'release_pos_x', 'release_pos_z', 'release_extension', 'arm_angle', 'outs_when_up', 'inning', 'inning_topbot', 'bat_score_diff', 'on_1b', 'on_2b', 'on_3b', 'pitch_cluster_id', 'pitch_cluster_local', 'umap_x', 'umap_y', 'launch_speed', 'launch_angle']
pitch_level: (719509, 2423)


,pitcher,batter,game_date,pitch_type,balls,strikes,plate_x,plate_z,release_speed,release_spin_rate,...,batter_events_group_other,batter_events_group_out,batter_events_group_reached,batter_events_group_walk,batter_description_group_ball,batter_description_group_foul,batter_description_group_inplay,batter_description_group_other,batter_description_group_strike,n_pitches_seen
0,592332,656811,2025-03-28,FF,1,2,-0.108750,4.016472,97.8,2418.0,...,0.755223,0.153951,0.000908,0.029064,0.373751,0.202543,0.168029,0.004087,0.251589,2202
1,592332,656811,2025-03-28,SL,1,2,-0.141998,2.795620,85.8,2232.0,...,0.755223,0.153951,0.000908,0.029064,0.373751,0.202543,0.168029,0.004087,0.251589,2202
2,592332,656811,2025-03-28,FF,1,1,-0.310140,3.007134,95.8,2203.0,...,0.755223,0.153951,0.000908,0.029064,0.373751,0.202543,0.168029,0.004087,0.251589,2202


범주형 간단 처리

In [ ]:
pitch_level["count"] = pitch_level["balls"].astype(str) + "-" + pitch_level["strikes"].astype(str)
pitch_level["matchup_id"] = pitch_level["pitcher"].astype(str) + "_" + pitch_level["batter"].astype(str)

pitch_level[["count","matchup_id"]].head(3)

pitch-level 저장

In [ ]:
pitch_level.to_parquet(OUT_PITCH_LEVEL, index=False)
print("saved:", OUT_PITCH_LEVEL, "shape:", pitch_level.shape)

B) Pair-level (투수-타자) 대응 요약 테이블 만들기

(pitcher, batter) 기본 집계: 투구수 + 결과 분포

In [7]:
# 기본 투구수
pair_base = pitch_level.groupby(["pitcher","batter"]).size().rename("n_pitches").reset_index()

# events_group 비율(간단)
ev = (
    pitch_level.groupby(["pitcher","batter","label_events_group"])
    .size().rename("n").reset_index()
)
ev = ev.merge(pair_base, on=["pitcher","batter"], how="left")
ev["ratio"] = ev["n"] / ev["n_pitches"]
ev_wide = ev.pivot(index=["pitcher","batter"], columns="label_events_group", values="ratio").fillna(0.0)
ev_wide.columns = [f"ev_{c}_ratio" for c in ev_wide.columns]

# desc_group 비율(간단)
dg = (
    pitch_level.groupby(["pitcher","batter","label_desc_group"])
    .size().rename("n").reset_index()
)
dg = dg.merge(pair_base, on=["pitcher","batter"], how="left")
dg["ratio"] = dg["n"] / dg["n_pitches"]
dg_wide = dg.pivot(index=["pitcher","batter"], columns="label_desc_group", values="ratio").fillna(0.0)
dg_wide.columns = [f"desc_{c}_ratio" for c in dg_wide.columns]

pair_level = pair_base.set_index(["pitcher","batter"]).join(ev_wide, how="left").join(dg_wide, how="left").fillna(0.0).reset_index()
print("pair_level:", pair_level.shape)
pair_level.head(5)

pair_level: (91493, 13)


,pitcher,batter,n_pitches,ev_hit_ratio,ev_other_ratio,ev_out_ratio,ev_reached_ratio,ev_walk_ratio,desc_ball_ratio,desc_foul_ratio,desc_inplay_ratio,desc_other_ratio,desc_strike_ratio
0,434378,457705,27,0.074074,0.814815,0.111111,0.000000,0.000000,0.333333,0.185185,0.148148,0.0,0.333333
1,434378,467793,14,0.000000,0.785714,0.071429,0.071429,0.071429,0.500000,0.142857,0.142857,0.0,0.214286
2,434378,502054,25,0.080000,0.800000,0.080000,0.000000,0.040000,0.480000,0.080000,0.120000,0.0,0.320000
3,434378,518692,10,0.100000,0.700000,0.200000,0.000000,0.000000,0.400000,0.100000,0.200000,0.0,0.300000
4,434378,542932,7,0.142857,0.714286,0.142857,0.000000,0.000000,0.285714,0.285714,0.285714,0.0,0.142857


투수 클러스터(구종 후보)별로 타자 성과 집계

(pitcher, batter, pitch_cluster_id) 단위로
투구수
hit/out/walk 비율
inplay/strike/ball 비율
평균 launch_speed/launch_angle (inplay 포함/전체 포함 선택 가능)

In [9]:
# 클러스터별 투구수
pcb = (
    pitch_level.groupby(["pitcher","batter","pitch_cluster_id"])
    .size().rename("n").reset_index()
)

# 클러스터별 hit/out/walk 집계
evc = (
    pitch_level.groupby(["pitcher","batter","pitch_cluster_id","label_events_group"])
    .size().rename("n_ev").reset_index()
)

# merge해서 ratio 만들기
evc = evc.merge(pcb, on=["pitcher","batter","pitch_cluster_id"], how="left")
evc["ratio"] = evc["n_ev"] / evc["n"]

# 클러스터별로 wide
evc_wide = evc.pivot(
    index=["pitcher","batter","pitch_cluster_id"],
    columns="label_events_group",
    values="ratio"
).fillna(0.0)

evc_wide.columns = [f"cl_ev_{c}_ratio" for c in evc_wide.columns]
evc_wide = evc_wide.reset_index()

# 클러스터별 inplay/strike/ball 비율도
dgc = (
    pitch_level.groupby(["pitcher","batter","pitch_cluster_id","label_desc_group"])
    .size().rename("n_dg").reset_index()
)
dgc = dgc.merge(pcb, on=["pitcher","batter","pitch_cluster_id"], how="left")
dgc["ratio"] = dgc["n_dg"] / dgc["n"]

dgc_wide = dgc.pivot(
    index=["pitcher","batter","pitch_cluster_id"],
    columns="label_desc_group",
    values="ratio"
).fillna(0.0)
dgc_wide.columns = [f"cl_desc_{c}_ratio" for c in dgc_wide.columns]
dgc_wide = dgc_wide.reset_index()

# 클러스터별 타구 지표 평균(전체 평균)
agg_cols = []
for c in ["launch_speed","launch_angle"]:
    if c in pitch_level.columns:
        agg_cols.append(c)

if agg_cols:
    cl_bip = (
        pitch_level.groupby(["pitcher","batter","pitch_cluster_id"])[agg_cols]
        .mean()
        .reset_index()
    )
    cl_bip = cl_bip.rename(columns={c: f"cl_avg_{c}" for c in agg_cols})
else:
    cl_bip = pd.DataFrame(columns=["pitcher","batter","pitch_cluster_id"])

# 클러스터별 집계 하나로 합치기
pair_cluster_level = pcb.merge(evc_wide, on=["pitcher","batter","pitch_cluster_id"], how="left")
pair_cluster_level = pair_cluster_level.merge(dgc_wide, on=["pitcher","batter","pitch_cluster_id"], how="left")
if not cl_bip.empty:
    pair_cluster_level = pair_cluster_level.merge(cl_bip, on=["pitcher","batter","pitch_cluster_id"], how="left")

pair_cluster_level = pair_cluster_level.fillna(0.0)

print("pair_cluster_level:", pair_cluster_level.shape)
pair_cluster_level.head(5)

pair_cluster_level: (219996, 16)


,pitcher,batter,pitch_cluster_id,n,cl_ev_hit_ratio,cl_ev_other_ratio,cl_ev_out_ratio,cl_ev_reached_ratio,cl_ev_walk_ratio,cl_desc_ball_ratio,cl_desc_foul_ratio,cl_desc_inplay_ratio,cl_desc_other_ratio,cl_desc_strike_ratio,cl_avg_launch_speed,cl_avg_launch_angle
0,434378,457705,434378_2,10,0.100000,0.600000,0.300,0.0,0.000,0.100000,0.200000,0.300000,0.0,0.400000,40.100000,10.500000
1,434378,457705,434378_4,9,0.111111,0.888889,0.000,0.0,0.000,0.333333,0.333333,0.111111,0.0,0.222222,26.355556,11.444444
2,434378,457705,434378_5,8,0.000000,1.000000,0.000,0.0,0.000,0.625000,0.000000,0.000000,0.0,0.375000,0.000000,0.000000
3,434378,467793,434378_1,2,0.000000,0.500000,0.000,0.5,0.000,0.500000,0.000000,0.500000,0.0,0.000000,44.400000,-7.500000
4,434378,467793,434378_3,8,0.000000,0.750000,0.125,0.0,0.125,0.500000,0.125000,0.125000,0.0,0.250000,22.462500,5.000000


pair_cluster_level을 pair_level에 “요약 피처”로 압축

In [10]:
TOPK = 5  # 쌍마다 상위 5개 클러스터만 요약

# 클러스터 비중(쌍 내에서) 계산
pair_cluster_level = pair_cluster_level.copy()
pair_cluster_level["cluster_ratio_in_pair"] = pair_cluster_level["n"] / pair_cluster_level.groupby(["pitcher","batter"])["n"].transform("sum")

# 상위 TOPK 클러스터만 뽑기
pair_cluster_level["rank_in_pair"] = pair_cluster_level.groupby(["pitcher","batter"])["cluster_ratio_in_pair"].rank(ascending=False, method="first")
top = pair_cluster_level[pair_cluster_level["rank_in_pair"] <= TOPK].copy()

# topK를 wide로 펼치기: cluster_id 자체도 저장 + 주요 성과 몇 개만 저장
keep_metrics = [c for c in top.columns if c.startswith("cl_ev_") or c.startswith("cl_desc_")] + [c for c in top.columns if c.startswith("cl_avg_")]
keep_metrics = sorted(set(keep_metrics))

rows = []
for (p,b), g in top.groupby(["pitcher","batter"]):
    g = g.sort_values("rank_in_pair")
    row = {"pitcher": int(p), "batter": int(b)}
    for i, (_, r) in enumerate(g.iterrows(), start=1):
        row[f"top{i}_cluster_id"] = r["pitch_cluster_id"]
        row[f"top{i}_cluster_ratio"] = float(r["cluster_ratio_in_pair"])
        # 대표 성과 몇 개
        for m in keep_metrics:
            row[f"top{i}_{m}"] = float(r[m])
    rows.append(row)

pair_topk = pd.DataFrame(rows)
print("pair_topk:", pair_topk.shape)
pair_topk.head(3)

pair_topk: (91493, 72)


,pitcher,batter,top1_cluster_id,top1_cluster_ratio,top1_cl_avg_launch_angle,top1_cl_avg_launch_speed,top1_cl_desc_ball_ratio,top1_cl_desc_foul_ratio,top1_cl_desc_inplay_ratio,top1_cl_desc_other_ratio,...,top5_cl_desc_ball_ratio,top5_cl_desc_foul_ratio,top5_cl_desc_inplay_ratio,top5_cl_desc_other_ratio,top5_cl_desc_strike_ratio,top5_cl_ev_hit_ratio,top5_cl_ev_other_ratio,top5_cl_ev_out_ratio,top5_cl_ev_reached_ratio,top5_cl_ev_walk_ratio
0,434378,457705,434378_2,0.370370,10.500000,40.100000,0.100000,0.200000,0.300000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,434378,467793,434378_3,0.571429,5.000000,22.462500,0.500000,0.125000,0.125000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,434378,502054,434378_2,0.520000,4.923077,23.061538,0.461538,0.153846,0.076923,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


pair_level + topK 요약 결합 + 저장

In [12]:
pair_level_final = pair_level.merge(pair_topk, on=["pitcher","batter"], how="left")

# 1) topK cluster_id 컬럼들은 문자열이므로 NaN은 "NONE" 같은 문자열로 채우기
cluster_id_cols = [c for c in pair_level_final.columns if c.startswith("top") and c.endswith("_cluster_id")]
for c in cluster_id_cols:
    pair_level_final[c] = pair_level_final[c].astype("string").fillna("NONE")

# 2) 나머지 숫자 컬럼만 0으로 채우기
num_cols = pair_level_final.select_dtypes(include=[np.number]).columns
pair_level_final[num_cols] = pair_level_final[num_cols].fillna(0.0)

# 3) 혹시 object로 남은 컬럼(대부분 문자열)은 NaN을 빈 문자열로
obj_cols = pair_level_final.select_dtypes(include=["object"]).columns
for c in obj_cols:
    pair_level_final[c] = pair_level_final[c].fillna("")

# 저장 (pyarrow가 꼬이면 engine="fastparquet"로 바꾸세요)
pair_level_final.to_parquet(OUT_PAIR_LEVEL, index=False)
print("saved:", OUT_PAIR_LEVEL, "shape:", pair_level_final.shape)

pair_level_final.head(5)

saved: C:\Users\Dalab-server\Documents\kangmin\Pitcheezy\data_preprocess\data\processed\matchup_pair_level.parquet shape: (91493, 83)


,pitcher,batter,n_pitches,ev_hit_ratio,ev_other_ratio,ev_out_ratio,ev_reached_ratio,ev_walk_ratio,desc_ball_ratio,desc_foul_ratio,...,top5_cl_desc_ball_ratio,top5_cl_desc_foul_ratio,top5_cl_desc_inplay_ratio,top5_cl_desc_other_ratio,top5_cl_desc_strike_ratio,top5_cl_ev_hit_ratio,top5_cl_ev_other_ratio,top5_cl_ev_out_ratio,top5_cl_ev_reached_ratio,top5_cl_ev_walk_ratio
0,434378,457705,27,0.074074,0.814815,0.111111,0.000000,0.000000,0.333333,0.185185,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,434378,467793,14,0.000000,0.785714,0.071429,0.071429,0.071429,0.500000,0.142857,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,434378,502054,25,0.080000,0.800000,0.080000,0.000000,0.040000,0.480000,0.080000,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,434378,518692,10,0.100000,0.700000,0.200000,0.000000,0.000000,0.400000,0.100000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,434378,542932,7,0.142857,0.714286,0.142857,0.000000,0.000000,0.285714,0.285714,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


C) 시각화 유틸

특정 (투수, 타자) matchup을 클러스터별로 시각화

In [13]:
def plot_pair_cluster_breakdown(pitcher_id: int, batter_id: int, min_n: int = 30):
    sub = pair_cluster_level[(pair_cluster_level["pitcher"] == pitcher_id) & (pair_cluster_level["batter"] == batter_id)].copy()
    if sub.empty:
        print("No data for this (pitcher, batter) pair.")
        return

    sub = sub.sort_values("n", ascending=False)
    if sub["n"].sum() < min_n:
        print(f"Total pitches in pair is {sub['n'].sum()} (<{min_n}). Plot may be noisy.")

    # x축: 클러스터
    x = np.arange(len(sub))

    # 기본: hit/out/walk 비율 (있을 때만)
    cols = [c for c in sub.columns if c.startswith("cl_ev_") and c.endswith("_ratio")]
    # 보기 좋게 3개만 우선(있으면)
    prefer = ["cl_ev_hit_ratio", "cl_ev_out_ratio", "cl_ev_walk_ratio"]
    cols_show = [c for c in prefer if c in cols]
    if not cols_show:
        cols_show = cols[:3]

    plt.figure()
    for c in cols_show:
        plt.plot(x, sub[c].values, marker="o", label=c.replace("cl_ev_","").replace("_ratio",""))
    plt.xticks(x, sub["pitch_cluster_id"].astype(str).values, rotation=90)
    plt.title(f"Pair (pitcher={pitcher_id}, batter={batter_id}) - outcome ratios by cluster")
    plt.legend()
    plt.show()

    # 클러스터 빈도도 같이 보기
    plt.figure()
    plt.bar(x, sub["n"].values)
    plt.xticks(x, sub["pitch_cluster_id"].astype(str).values, rotation=90)
    plt.title(f"Pair (pitcher={pitcher_id}, batter={batter_id}) - pitches per cluster")
    plt.show()

# 사용 예시:
# plot_pair_cluster_breakdown(592332, 807713)

특정 투수 기준으로 “타자들 상대로 클러스터별 성과” 히트맵 준비

In [15]:
def pitcher_cluster_heatmap_data(pitcher_id: int, metric: str = "cl_ev_hit_ratio", min_pair_pitches: int = 30, top_batters: int = 30):
    # 특정 투수의 pair_cluster_level에서, 쌍 투구수가 충분한 타자만
    sub = pair_cluster_level[pair_cluster_level["pitcher"] == pitcher_id].copy()
    if sub.empty:
        print("No data for pitcher.")
        return None

    # (pitcher,batter) 전체 투구수가 min_pair_pitches 이상인 타자만
    pair_n = sub.groupby("batter")["n"].sum()
    keep_b = pair_n[pair_n >= min_pair_pitches].sort_values(ascending=False).head(top_batters).index.tolist()
    sub = sub[sub["batter"].isin(keep_b)]

    if metric not in sub.columns:
        print("metric not found:", metric)
        return None

    # batter x cluster matrix
    mat = sub.pivot_table(index="batter", columns="pitch_cluster_id", values=metric, aggfunc="mean").fillna(0.0)
    return mat

# 사용 예시:
# mat = pitcher_cluster_heatmap_data(592332, metric="cl_ev_hit_ratio")
# if mat is not None:
#     plt.figure()
#     plt.imshow(mat.values, aspect="auto")
#     plt.title("batter x cluster heatmap (hit ratio)")
#     plt.show()